In [9]:
import pandas as pd
import numpy as np
import yfinance as yf
import datetime
from ta.momentum import RSIIndicator, StochasticOscillator, ROCIndicator
from ta.trend import EMAIndicator, MACD, SMAIndicator, ADXIndicator
from ta.volatility import AverageTrueRange, BollingerBands
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score, recall_score
from prophet import Prophet
from statsmodels.tsa.holtwinters import ExponentialSmoothing
import warnings
warnings.filterwarnings("ignore")

In [14]:
print("📥 Downloading gold data...")
df = yf.download('GC=F', period="730d", interval='1h', ignore_tz=True, progress=False)

# Save for reference
now = datetime.datetime.now()
filename = f"/Users/mac/Desktop/Upwork/GoldSignalGenerator/Artifacts/{now}_GLD.csv"
df.to_csv(filename)
print(f"Data saved to {filename}")

📥 Downloading gold data...
Data saved to /Users/mac/Desktop/Upwork/GoldSignalGenerator/Artifacts/2025-10-09 14:00:26.779243_GLD.csv


In [10]:
df = pd.read_csv("/Users/mac/Desktop/Upwork/GoldSignalGenerator/Artifacts/2025-10-09 00:07:45.956111_GLD.csv", skiprows=[1,2])

df.rename(columns={"Price": "Datetime"}, inplace=True)
df["Datetime"] = pd.to_datetime(df["Datetime"], errors="coerce")
df.rename(columns={"Datetime": "ds"}, inplace=True)

# Forward fill
df['Close'] = df['Close'].ffill()
df['High'] = df['High'].ffill()
df['Low'] = df['Low'].ffill()

df.head()

,ds,Close,High,Low,Open,Volume
0,2023-05-18 00:00:00,1982.500000,1985.099976,1982.199951,1985.099976,0
1,2023-05-18 01:00:00,1980.800049,1982.500000,1980.000000,1982.500000,4884
2,2023-05-18 02:00:00,1979.900024,1984.199951,1979.599976,1980.800049,6725
3,2023-05-18 03:00:00,1977.000000,1982.000000,1976.000000,1979.800049,12013
4,2023-05-18 04:00:00,1977.900024,1978.699951,1974.599976,1977.099976,11054


In [11]:
def add_prophet_features(df):
    """Add Prophet forecasting features"""
    # Prepare data for Prophet
    prophet_df = df[['ds', 'Close']].copy()
    prophet_df.columns = ['ds', 'y']
    prophet_df['ds'] = pd.to_datetime(prophet_df['ds'])
    
    # Train Prophet model
    prophet_model = Prophet(
        growth='linear',
        seasonality_mode='additive',
        interval_width=0.95,
        daily_seasonality=True,
        weekly_seasonality=True,
        yearly_seasonality=False,
        changepoint_prior_scale=0.05
    )
    
    prophet_model.fit(prophet_df)
    
    # Predict only for existing timestamps
    forecast = prophet_model.predict(prophet_df[['ds']])
    
    # Add Prophet features (flatten to 1D)
    df['prophet_yhat'] = forecast['yhat'].values.ravel()
    df['prophet_trend'] = forecast['trend'].values.ravel()
    df['prophet_upper'] = forecast['yhat_upper'].values.ravel()
    df['prophet_lower'] = forecast['yhat_lower'].values.ravel()
    
    # Derived features
    df['prophet_diff'] = df['Close'].values.ravel() - df['prophet_yhat'].values.ravel()
    df['prophet_trend_strength'] = df['prophet_trend'].diff().values.ravel()
    df['prophet_uncertainty'] = df['prophet_upper'].values.ravel() - df['prophet_lower'].values.ravel()
    
    return df


df = add_prophet_features(df)
print("✅ Prophet features added")


13:46:04 - cmdstanpy - INFO - Chain [1] start processing
13:46:10 - cmdstanpy - INFO - Chain [1] done processing


✅ Prophet features added


In [12]:
print("\n📊 Adding ETS features...")

def add_ets_features(df):
    """Add Exponential Smoothing features"""
    try:
        # Multiplicative ETS
        ets_mult = ExponentialSmoothing(
            df['Close'], 
            trend='mul', 
            seasonal='mul', 
            seasonal_periods=24*7  # Weekly seasonality for hourly data
        ).fit()
        df['ets_multiplicative'] = ets_mult.fittedvalues.values.ravel()
        df['ets_mult_resid'] = (df['Close'].values - df['ets_multiplicative'].values).ravel()
        
    except:
        print("⚠️ Multiplicative ETS failed, using simpler model")
        ets_mult = ExponentialSmoothing(
            df['Close'], 
            trend='add', 
            seasonal=None
        ).fit()
        df['ets_multiplicative'] = ets_mult.fittedvalues.values.ravel()
        df['ets_mult_resid'] = (df['Close'].values - df['ets_multiplicative'].values).ravel()
    
    try:
        # Additive ETS
        ets_add = ExponentialSmoothing(
            df['Close'], 
            trend='add', 
            seasonal='add', 
            seasonal_periods=24*7
        ).fit()
        df['ets_additive'] = ets_add.fittedvalues.values.ravel()
        df['ets_add_resid'] = (df['Close'].values - df['ets_additive'].values).ravel()
        
    except:
        print("⚠️ Additive ETS failed, using simpler model")
        ets_add = ExponentialSmoothing(
            df['Close'], 
            trend='add', 
            seasonal=None
        ).fit()
        df['ets_additive'] = ets_add.fittedvalues.values.ravel()
        df['ets_add_resid'] = (df['Close'].values - df['ets_additive'].values).ravel()
    
    # Derived features
    df['ets_trend_agreement'] = (
        (df['ets_multiplicative'] > df['ets_multiplicative'].shift(1)) & 
        (df['ets_additive'] > df['ets_additive'].shift(1))
    ).astype(int)
    
    return df

try:
    df = add_ets_features(df)
    print("✅ ETS features added")
except Exception as e:
    print(f"⚠️ ETS failed: {e}")

# =============================================
# 4. STANDARD TECHNICAL INDICATORS
# =============================================
print("\n🔧 Creating standard features...")

# Rename ds back to Datetime
df.rename(columns={"ds": "Datetime"}, inplace=True)
df['Datetime'] = pd.to_datetime(df['Datetime'])

# Returns
df['return'] = df['Close'].pct_change()
df['return_abs'] = df['return'].abs()
df['hl_spread'] = (df['High'] - df['Low']) / df['Close']

# EMAs
for window in [9, 12, 21, 26, 50]:
    ema_values = EMAIndicator(df['Close'], window=window).ema_indicator()
    df[f'ema_{window}'] = ema_values.values.ravel()

df['ema_fast'] = df['ema_12']
df['ema_slow'] = df['ema_26']
df['ema_cross'] = (df['ema_fast'] > df['ema_slow']).astype(int)
df['ema_distance'] = (df['ema_fast'] - df['ema_slow']) / df['Close']

# Price vs EMAs
df['price_vs_ema9'] = (df['Close'] - df['ema_9']) / df['Close']
df['price_vs_ema21'] = (df['Close'] - df['ema_21']) / df['Close']
df['price_vs_ema50'] = (df['Close'] - df['ema_50']) / df['Close']

# SMAs
for window in [20, 50, 100, 200]:
    sma_values = SMAIndicator(df['Close'], window=window).sma_indicator()
    df[f'sma_{window}'] = sma_values.values.ravel()

# Trend features
df['price_above_sma50'] = (df['Close'] > df['sma_50']).astype(int)
df['price_above_sma200'] = (df['Close'] > df['sma_200']).astype(int)
df['sma50_above_sma200'] = (df['sma_50'] > df['sma_200']).astype(int)

# RSI
rsi_indicator = RSIIndicator(df['Close'], window=14)
df['rsi'] = rsi_indicator.rsi().values.ravel()
df['rsi_ma'] = df['rsi'].rolling(5).mean()
df['rsi_overbought'] = (df['rsi'] > 70).astype(int)
df['rsi_oversold'] = (df['rsi'] < 30).astype(int)
df['rsi_bullish'] = (df['rsi'] > 50).astype(int)

# MACD
macd = MACD(df['Close'])
df['macd'] = macd.macd()
df['macd_signal'] = macd.macd_signal()
df['macd_histogram'] = df['macd'] - df['macd_signal']
df['macd_cross'] = (df['macd'] > df['macd_signal']).astype(int)
df['macd_bullish'] = (df['macd'] > 0).astype(int)

# ADX
adx = ADXIndicator(df['High'], df['Low'], df['Close'], window=14)
df['adx'] = adx.adx()
df['adx_pos'] = adx.adx_pos()
df['adx_neg'] = adx.adx_neg()
df['adx_bullish'] = (df['adx_pos'] > df['adx_neg']).astype(int)

# ATR
df['ATR'] = AverageTrueRange(df['High'], df['Low'], df['Close'], window=14).average_true_range()
df['atr_pct'] = df['ATR'] / df['Close']
df['atr_ma'] = df['ATR'].rolling(10).mean()

# Stochastic
stoch = StochasticOscillator(df['High'], df['Low'], df['Close'])
df['stoch_k'] = stoch.stoch()
df['stoch_d'] = stoch.stoch_signal()
df['stoch_bullish'] = (df['stoch_k'] > 50).astype(int)

# ROC
df['roc'] = ROCIndicator(df['Close'], window=12).roc()
df['roc_positive'] = (df['roc'] > 0).astype(int)

# Bollinger Bands
bb = BollingerBands(df['Close'], window=20, window_dev=2)
df['bb_high'] = bb.bollinger_hband()
df['bb_low'] = bb.bollinger_lband()
df['bb_mid'] = bb.bollinger_mavg()
df['bb_width'] = (df['bb_high'] - df['bb_low']) / df['bb_mid']
df['bb_position'] = (df['Close'] - df['bb_low']) / (df['bb_high'] - df['bb_low'])

# Momentum
df['price_momentum_12'] = df['Close'].pct_change(periods=12)
df['price_momentum_24'] = df['Close'].pct_change(periods=24)
df['momentum_positive'] = (df['price_momentum_12'] > 0).astype(int)

# Support/Resistance
df['resistance_20'] = df['High'].rolling(20).max()
df['support_20'] = df['Low'].rolling(20).min()
df['price_in_range'] = (df['Close'] - df['support_20']) / (df['resistance_20'] - df['support_20'])

# Lag features
for lag in [1, 2, 3, 6, 12, 24]:
    df[f'return_lag_{lag}'] = df['return'].shift(lag)
    df[f'rsi_lag_{lag}'] = df['rsi'].shift(lag)

# Rolling statistics
df['return_std_12'] = df['return'].rolling(12).std()
df['return_mean_12'] = df['return'].rolling(12).mean()

# TREND SCORE
df['trend_score'] = (
    df['price_above_sma50'] + 
    df['price_above_sma200'] + 
    df['sma50_above_sma200'] +
    df['ema_cross'] +
    df['macd_bullish'] +
    df['adx_bullish'] +
    df['rsi_bullish'] +
    df['momentum_positive'] +
    df['roc_positive']
)

# Time features
df['hour'] = df['Datetime'].dt.hour
df['day_of_week'] = df['Datetime'].dt.dayofweek
df['is_london_session'] = ((df['hour'] >= 8) & (df['hour'] <= 16)).astype(int)
df['is_ny_session'] = ((df['hour'] >= 13) & (df['hour'] <= 21)).astype(int)
df['is_asian_session'] = ((df['hour'] >= 0) & (df['hour'] <= 8)).astype(int)

df = df.dropna().reset_index(drop=True)
print(f"After features: {len(df)} rows with {len(df.columns)} columns")

# =============================================
# 5. CREATE LABELS WITH MULTIPLE STRATEGIES
# =============================================
print("\n📊 Creating labels - trying multiple strategies...")

# STRATEGY 1: Your exact backtest (may give few signals)
def create_labels_backtest(df, horizon=6, threshold=0.01, rr_ratio=2.0, risk_mult=1.5):
    df = df.copy().reset_index(drop=True)
    
    df['potential_entry'] = df['Close']
    df['potential_sl_buy'] = df['Close'] - risk_mult * df['ATR']
    df['potential_tp_buy'] = df['Close'] + rr_ratio * risk_mult * df['ATR']
    df['potential_sl_sell'] = df['Close'] + risk_mult * df['ATR']
    df['potential_tp_sell'] = df['Close'] - rr_ratio * risk_mult * df['ATR']
    
    df['future_return'] = (df['Close'].shift(-horizon) - df['Close']) / df['Close']
    
    signal = np.zeros(len(df), dtype=int)
    
    for i in range(len(df) - horizon):
        future_return = df.loc[i, 'future_return']
        
        if pd.isna(future_return):
            continue
        
        future = df.loc[i+1:i+horizon]
        if future.empty:
            continue
        
        sl_buy = df.loc[i, 'potential_sl_buy']
        tp_buy = df.loc[i, 'potential_tp_buy']
        sl_sell = df.loc[i, 'potential_sl_sell']
        tp_sell = df.loc[i, 'potential_tp_sell']
        
        # BUY
        tp_hit_buy = (future['High'] >= tp_buy).any()
        sl_hit_buy = (future['Low'] <= sl_buy).any()
        
        if tp_hit_buy:
            if sl_hit_buy:
                tp_idx = future[future['High'] >= tp_buy].index[0]
                sl_idx = future[future['Low'] <= sl_buy].index[0]
                if tp_idx < sl_idx and future_return > threshold:
                    signal[i] = 1
            elif future_return > threshold:
                signal[i] = 1
        
        # SELL
        if signal[i] == 0:
            tp_hit_sell = (future['Low'] <= tp_sell).any()
            sl_hit_sell = (future['High'] >= sl_sell).any()
            
            if tp_hit_sell:
                if sl_hit_sell:
                    tp_idx = future[future['Low'] <= tp_sell].index[0]
                    sl_idx = future[future['High'] >= sl_sell].index[0]
                    if tp_idx < sl_idx and future_return < -threshold:
                        signal[i] = 2
                elif future_return < -threshold:
                    signal[i] = 2
    
    df['signal'] = signal
    return df

# Try strategy 1
print("Strategy 1: Exact backtest parameters (Horizon=6, Threshold=1%)")
df = create_labels_backtest(df, horizon=6, threshold=0.01, rr_ratio=2.0, risk_mult=1.5)

buy_count = (df['signal'] == 1).sum()
sell_count = (df['signal'] == 2).sum()

print(f"BUY: {buy_count}, SELL: {sell_count}")

# If too few signals, try strategy 2
if buy_count < 400 or sell_count < 400:
    print("\n⚠️ Too few signals, trying Strategy 2...")
    print("Strategy 2: Relaxed threshold (Horizon=6, Threshold=0.5%)")
    
    df = create_labels_backtest(df, horizon=6, threshold=0.005, rr_ratio=2.0, risk_mult=1.5)
    
    buy_count = (df['signal'] == 1).sum()
    sell_count = (df['signal'] == 2).sum()
    print(f"BUY: {buy_count}, SELL: {sell_count}")

# If still too few, try strategy 3
if buy_count < 400 or sell_count < 400:
    print("\n⚠️ Still too few signals, trying Strategy 3...")
    print("Strategy 3: Even more relaxed (Horizon=6, Threshold=0.3%)")
    
    df = create_labels_backtest(df, horizon=6, threshold=0.003, rr_ratio=2.0, risk_mult=1.5)
    
    buy_count = (df['signal'] == 1).sum()
    sell_count = (df['signal'] == 2).sum()
    print(f"BUY: {buy_count}, SELL: {sell_count}")

print("\n✅ Final signal distribution:")
print(df['signal'].value_counts())
print("\nPercentages:")
print(df['signal'].value_counts(normalize=True) * 100)

# =============================================
# REST OF YOUR CODE STAYS THE SAME
# =============================================
exclude_cols = ['Datetime', 'signal', 'potential_entry', 'potential_sl_buy', 
                'potential_tp_buy', 'potential_sl_sell', 'potential_tp_sell',
                'future_return']

feature_cols = [col for col in df.columns if col not in exclude_cols]
print(f"\n📦 Using {len(feature_cols)} features")

train_size = int(0.75 * len(df))
train = df.iloc[:train_size].copy()
test = df.iloc[train_size:].copy()

X_train = train[feature_cols]
y_train = train['signal']
X_test = test[feature_cols]
y_test = test['signal']

print(f"Training: {len(X_train)} samples")
print(f"Test: {len(X_test)} samples")
print("\nTraining signals:")
print(y_train.value_counts())

# Continue with your binary models training...
print("\n🤖 Training separate binary models...")

from xgboost import XGBClassifier

y_train_buy = (y_train == 1).astype(int)
y_test_buy = (y_test == 1).astype(int)
buy_scale = (y_train_buy == 0).sum() / (y_train_buy == 1).sum()

buy_model = XGBClassifier(
    n_estimators=1500,
    max_depth=5,
    learning_rate=0.015,
    subsample=0.8,
    colsample_bytree=0.7,
    min_child_weight=3,
    gamma=0.3,
    reg_alpha=0.5,
    reg_lambda=1.0,
    scale_pos_weight=buy_scale,
    random_state=42
)

buy_model.fit(X_train, y_train_buy)
print("✅ BUY model trained")

y_train_sell = (y_train == 2).astype(int)
y_test_sell = (y_test == 2).astype(int)
sell_scale = (y_train_sell == 0).sum() / (y_train_sell == 1).sum()

sell_model = XGBClassifier(
    n_estimators=1500,
    max_depth=5,
    learning_rate=0.015,
    subsample=0.8,
    colsample_bytree=0.7,
    min_child_weight=3,
    gamma=0.3,
    reg_alpha=0.5,
    reg_lambda=1.0,
    scale_pos_weight=sell_scale,
    random_state=42
)

sell_model.fit(X_train, y_train_sell)
print("✅ SELL model trained")

print("\n🎯 Finding optimal thresholds...")

buy_proba = buy_model.predict_proba(X_test)[:, 1]
sell_proba = sell_model.predict_proba(X_test)[:, 1]

def find_best_threshold(y_true_binary, y_proba, target_precision=0.75, min_signals=5):
    best_threshold = 0.5
    best_score = 0
    best_metrics = None
    
    for thresh in np.linspace(0.30, 0.90, 300):
        y_pred = (y_proba >= thresh).astype(int)
        
        n_signals = y_pred.sum()
        if n_signals < min_signals:
            continue
        
        from sklearn.metrics import precision_score, recall_score
        precision = precision_score(y_true_binary, y_pred, zero_division=0)
        recall = recall_score(y_true_binary, y_pred, zero_division=0)
        
        if precision < target_precision:
            continue
        
        score = precision * np.sqrt(recall) * np.log(n_signals + 1)
        
        if score > best_score:
            best_score = score
            best_threshold = thresh
            best_metrics = {
                'threshold': thresh,
                'precision': precision,
                'recall': recall,
                'signals': n_signals
            }
    
    return best_threshold, best_metrics

# Try BUY with decreasing precision targets
for target in [0.80, 0.75, 0.70, 0.65]:
    buy_threshold, buy_metrics = find_best_threshold(y_test_buy, buy_proba, target_precision=target, min_signals=5)
    if buy_metrics:
        print(f"\n🟢 BUY Model (target {target*100:.0f}%):")
        print(f"   Threshold: {buy_metrics['threshold']:.3f}")
        print(f"   Precision: {buy_metrics['precision']*100:.1f}% ✅")
        print(f"   Recall: {buy_metrics['recall']*100:.1f}%")
        print(f"   Signals: {buy_metrics['signals']}")
        break
else:
    print("\n🟢 BUY Model: Using default threshold 0.50")
    buy_threshold = 0.50

# Try SELL with decreasing precision targets
for target in [0.80, 0.75, 0.70, 0.65, 0.60]:
    sell_threshold, sell_metrics = find_best_threshold(y_test_sell, sell_proba, target_precision=target, min_signals=5)
    if sell_metrics:
        print(f"\n🔴 SELL Model (target {target*100:.0f}%):")
        print(f"   Threshold: {sell_metrics['threshold']:.3f}")
        print(f"   Precision: {sell_metrics['precision']*100:.1f}% ✅")
        print(f"   Recall: {sell_metrics['recall']*100:.1f}%")
        print(f"   Signals: {sell_metrics['signals']}")
        break
else:
    print("\n🔴 SELL Model: Using default threshold 0.50")
    sell_threshold = 0.50




📊 Adding ETS features...
✅ ETS features added

🔧 Creating standard features...
After features: 13560 rows with 93 columns

📊 Creating labels - trying multiple strategies...
Strategy 1: Exact backtest parameters (Horizon=6, Threshold=1%)
BUY: 288, SELL: 246

⚠️ Too few signals, trying Strategy 2...
Strategy 2: Relaxed threshold (Horizon=6, Threshold=0.5%)
BUY: 770, SELL: 707

✅ Final signal distribution:
signal
0    12083
1      770
2      707
Name: count, dtype: int64

Percentages:
signal
0    89.107670
1     5.678466
2     5.213864
Name: proportion, dtype: float64

📦 Using 92 features
Training: 10170 samples
Test: 3390 samples

Training signals:
signal
0    9121
1     548
2     501
Name: count, dtype: int64

🤖 Training separate binary models...
✅ BUY model trained
✅ SELL model trained

🎯 Finding optimal thresholds...

🟢 BUY Model: Using default threshold 0.50

🔴 SELL Model: Using default threshold 0.50


In [13]:
# =============================================
# EVALUATION
# =============================================
print("\n📊 EVALUATING MODELS...")
print("=" * 70)

from sklearn.metrics import precision_score, recall_score, classification_report, confusion_matrix

# Get predictions
buy_proba = buy_model.predict_proba(X_test)[:, 1]
sell_proba = sell_model.predict_proba(X_test)[:, 1]

# Apply thresholds
y_pred_buy = (buy_proba >= buy_threshold).astype(int)
y_pred_sell = (sell_proba >= sell_threshold).astype(int)

# Calculate metrics for BUY
buy_precision = precision_score(y_test_buy, y_pred_buy, zero_division=0)
buy_recall = recall_score(y_test_buy, y_pred_buy, zero_division=0)
buy_signals_generated = y_pred_buy.sum()
buy_correct = ((y_pred_buy == 1) & (y_test_buy == 1)).sum()

print("\n🟢 BUY MODEL PERFORMANCE:")
print(f"   Precision: {buy_precision*100:.1f}%")
print(f"   Recall: {buy_recall*100:.1f}%")
print(f"   Signals generated: {buy_signals_generated}")
print(f"   Correct predictions: {buy_correct}")
print(f"   Accuracy: {buy_correct}/{buy_signals_generated} = {buy_correct/buy_signals_generated*100:.1f}%" if buy_signals_generated > 0 else "")

# Calculate metrics for SELL
sell_precision = precision_score(y_test_sell, y_pred_sell, zero_division=0)
sell_recall = recall_score(y_test_sell, y_pred_sell, zero_division=0)
sell_signals_generated = y_pred_sell.sum()
sell_correct = ((y_pred_sell == 1) & (y_test_sell == 1)).sum()

print("\n🔴 SELL MODEL PERFORMANCE:")
print(f"   Precision: {sell_precision*100:.1f}%")
print(f"   Recall: {sell_recall*100:.1f}%")
print(f"   Signals generated: {sell_signals_generated}")
print(f"   Correct predictions: {sell_correct}")
print(f"   Accuracy: {sell_correct}/{sell_signals_generated} = {sell_correct/sell_signals_generated*100:.1f}%" if sell_signals_generated > 0 else "")

# Overall statistics
total_signals = buy_signals_generated + sell_signals_generated
avg_precision = (buy_precision + sell_precision) / 2

print("\n📊 OVERALL STATISTICS:")
print(f"   Total trading signals: {total_signals}")
print(f"   Average precision: {avg_precision*100:.1f}%")
print(f"   Signal frequency: {total_signals / len(y_test) * 100:.1f}% of test candles")

# Confusion matrices
print("\n📋 BUY Confusion Matrix:")
buy_cm = confusion_matrix(y_test_buy, y_pred_buy)
print(buy_cm)
print("   [[True Neg, False Pos],")
print("    [False Neg, True Pos]]")

print("\n📋 SELL Confusion Matrix:")
sell_cm = confusion_matrix(y_test_sell, y_pred_sell)
print(sell_cm)

# =============================================
# SHOW SAMPLE SIGNALS
# =============================================
print("\n\n🎯 SAMPLE SIGNALS FROM TEST SET:")
print("=" * 70)

# Create results dataframe
test_results = test.copy()
test_results['buy_proba'] = buy_proba
test_results['sell_proba'] = sell_proba
test_results['predicted_signal'] = 'HOLD'
test_results.loc[y_pred_buy == 1, 'predicted_signal'] = 'BUY'
test_results.loc[y_pred_sell == 1, 'predicted_signal'] = 'SELL'

# Show BUY signals
buy_signals = test_results[test_results['predicted_signal'] == 'BUY'].head(10)
print(f"\n🟢 SAMPLE BUY SIGNALS (Showing 10/{buy_signals_generated}):\n")

for idx, row in buy_signals.iterrows():
    actual = "✅ CORRECT" if row['signal'] == 1 else "❌ WRONG"
    print(f"Time: {row['Datetime']} | Confidence: {row['buy_proba']*100:.1f}% | {actual}")
    print(f"  Entry: {row['Close']:.2f} | SL: {row['potential_sl_buy']:.2f} | TP: {row['potential_tp_buy']:.2f}")
    print()

# Show SELL signals
sell_signals = test_results[test_results['predicted_signal'] == 'SELL'].head(10)
print(f"\n🔴 SAMPLE SELL SIGNALS (Showing 10/{sell_signals_generated}):\n")

for idx, row in sell_signals.iterrows():
    actual = "✅ CORRECT" if row['signal'] == 2 else "❌ WRONG"
    print(f"Time: {row['Datetime']} | Confidence: {row['sell_proba']*100:.1f}% | {actual}")
    print(f"  Entry: {row['Close']:.2f} | SL: {row['potential_sl_sell']:.2f} | TP: {row['potential_tp_sell']:.2f}")
    print()

# =============================================
# IMPROVE THRESHOLDS IF NEEDED
# =============================================
print("\n🔧 THRESHOLD OPTIMIZATION:")
print("=" * 70)

# Try to find better thresholds
print("\nSearching for optimal thresholds...")

def optimize_threshold(y_true_binary, y_proba, target_precision=0.70, min_signals=10):
    results = []
    
    for thresh in np.linspace(0.3, 0.9, 150):
        y_pred = (y_proba >= thresh).astype(int)
        
        n_signals = y_pred.sum()
        if n_signals < min_signals:
            continue
        
        precision = precision_score(y_true_binary, y_pred, zero_division=0)
        recall = recall_score(y_true_binary, y_pred, zero_division=0)
        
        results.append({
            'threshold': thresh,
            'precision': precision,
            'recall': recall,
            'f1': 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0,
            'signals': n_signals
        })
    
    if not results:
        return None
    
    df_results = pd.DataFrame(results)
    
    # Find best threshold above target precision
    good = df_results[df_results['precision'] >= target_precision]
    
    if len(good) > 0:
        # Among good precision, maximize recall
        best = good.loc[good['recall'].idxmax()]
    else:
        # If can't reach target, maximize F1
        best = df_results.loc[df_results['f1'].idxmax()]
    
    return best

# Optimize BUY
buy_optimal = optimize_threshold(y_test_buy, buy_proba, target_precision=0.70, min_signals=10)

if buy_optimal is not None:
    print(f"\n🟢 OPTIMIZED BUY THRESHOLD:")
    print(f"   Threshold: {buy_optimal['threshold']:.3f}")
    print(f"   Precision: {buy_optimal['precision']*100:.1f}%")
    print(f"   Recall: {buy_optimal['recall']*100:.1f}%")
    print(f"   F1 Score: {buy_optimal['f1']:.3f}")
    print(f"   Signals: {int(buy_optimal['signals'])}")
    
    # Update if better than current
    if buy_optimal['precision'] > buy_precision:
        print(f"   ✅ Better than current ({buy_precision*100:.1f}%)")
        buy_threshold = buy_optimal['threshold']
    else:
        print(f"   ⚠️ Current threshold is already good")
else:
    print("\n🟢 Could not optimize BUY threshold")

# Optimize SELL
sell_optimal = optimize_threshold(y_test_sell, sell_proba, target_precision=0.70, min_signals=10)

if sell_optimal is not None:
    print(f"\n🔴 OPTIMIZED SELL THRESHOLD:")
    print(f"   Threshold: {sell_optimal['threshold']:.3f}")
    print(f"   Precision: {sell_optimal['precision']*100:.1f}%")
    print(f"   Recall: {sell_optimal['recall']*100:.1f}%")
    print(f"   F1 Score: {sell_optimal['f1']:.3f}")
    print(f"   Signals: {int(sell_optimal['signals'])}")
    
    if sell_optimal['precision'] > sell_precision:
        print(f"   ✅ Better than current ({sell_precision*100:.1f}%)")
        sell_threshold = sell_optimal['threshold']
    else:
        print(f"   ⚠️ Current threshold is already good")
else:
    print("\n🔴 Could not optimize SELL threshold")

# =============================================
# FINAL SUMMARY
# =============================================
print("\n\n" + "=" * 70)
print("🎉 FINAL MODEL SUMMARY")
print("=" * 70)

# Re-evaluate with optimized thresholds
y_pred_buy_final = (buy_proba >= buy_threshold).astype(int)
y_pred_sell_final = (sell_proba >= sell_threshold).astype(int)

buy_precision_final = precision_score(y_test_buy, y_pred_buy_final, zero_division=0)
buy_recall_final = recall_score(y_test_buy, y_pred_buy_final, zero_division=0)
buy_signals_final = y_pred_buy_final.sum()

sell_precision_final = precision_score(y_test_sell, y_pred_sell_final, zero_division=0)
sell_recall_final = recall_score(y_test_sell, y_pred_sell_final, zero_division=0)
sell_signals_final = y_pred_sell_final.sum()

print(f"\n🟢 BUY:")
print(f"   Precision: {buy_precision_final*100:.1f}% {'✅' if buy_precision_final >= 0.70 else '⚠️'}")
print(f"   Recall: {buy_recall_final*100:.1f}%")
print(f"   Threshold: {buy_threshold:.3f}")
print(f"   Signals: {buy_signals_final}")

print(f"\n🔴 SELL:")
print(f"   Precision: {sell_precision_final*100:.1f}% {'✅' if sell_precision_final >= 0.70 else '⚠️'}")
print(f"   Recall: {sell_recall_final*100:.1f}%")
print(f"   Threshold: {sell_threshold:.3f}")
print(f"   Signals: {sell_signals_final}")

print(f"\n📊 Overall:")
print(f"   Average Precision: {(buy_precision_final + sell_precision_final)/2*100:.1f}%")
print(f"   Total Signals: {buy_signals_final + sell_signals_final}")

# Save updated model with optimized thresholds
model_data['buy_threshold'] = buy_threshold
model_data['sell_threshold'] = sell_threshold

joblib.dump(model_data, '/Users/mac/Desktop/Upwork/GoldSignalGenerator/Artifacts/GLD_FINAL_optimized.pkl')
print("\n💾 Updated model saved as 'GLD_FINAL_optimized.pkl'")
print("✅ Evaluation complete!")


📊 EVALUATING MODELS...

🟢 BUY MODEL PERFORMANCE:
   Precision: 11.1%
   Recall: 0.5%
   Signals generated: 9
   Correct predictions: 1
   Accuracy: 1/9 = 11.1%

🔴 SELL MODEL PERFORMANCE:
   Precision: 6.6%
   Recall: 4.9%
   Signals generated: 151
   Correct predictions: 10
   Accuracy: 10/151 = 6.6%

📊 OVERALL STATISTICS:
   Total trading signals: 160
   Average precision: 8.9%
   Signal frequency: 4.7% of test candles

📋 BUY Confusion Matrix:
[[3160    8]
 [ 221    1]]
   [[True Neg, False Pos],
    [False Neg, True Pos]]

📋 SELL Confusion Matrix:
[[3043  141]
 [ 196   10]]


🎯 SAMPLE SIGNALS FROM TEST SET:

🟢 SAMPLE BUY SIGNALS (Showing 10/9):

Time: 2025-08-14 03:00:00 | Confidence: 55.0% | ❌ WRONG
  Entry: 3394.40 | SL: 3382.08 | TP: 3419.04

Time: 2025-08-19 07:00:00 | Confidence: 55.7% | ❌ WRONG
  Entry: 3384.40 | SL: 3376.19 | TP: 3400.82

Time: 2025-08-19 08:00:00 | Confidence: 55.8% | ❌ WRONG
  Entry: 3379.40 | SL: 3371.11 | TP: 3395.98

Time: 2025-08-19 09:00:00 | Confidenc

NameError: name 'model_data' is not defined

In [40]:
def add_ets_features(df):
    """Add Exponential Smoothing features"""
    try:
        # Multiplicative ETS
        ets_mult = ExponentialSmoothing(
            df['Close'], 
            trend='mul', 
            seasonal='mul', 
            seasonal_periods=24*7  # Weekly seasonality for hourly data
        ).fit()
        df['ets_multiplicative'] = ets_mult.fittedvalues
        df['ets_mult_resid'] = df['Close'] - df['ets_multiplicative']
        
    except:
        print("⚠️ Multiplicative ETS failed, using simpler model")
        ets_mult = ExponentialSmoothing(
            df['Close'], 
            trend='add', 
            seasonal=None
        ).fit()
        df['ets_multiplicative'] = ets_mult.fittedvalues
        df['ets_mult_resid'] = df['Close'] - df['ets_multiplicative']
    
    try:
        # Additive ETS
        ets_add = ExponentialSmoothing(
            df['Close'], 
            trend='add', 
            seasonal='add', 
            seasonal_periods=24*7
        ).fit()
        df['ets_additive'] = ets_add.fittedvalues
        df['ets_add_resid'] = df['Close'] - df['ets_additive']
        
    except:
        print("⚠️ Additive ETS failed, using simpler model")
        ets_add = ExponentialSmoothing(
            df['Close'], 
            trend='add', 
            seasonal=None
        ).fit()
        df['ets_additive'] = ets_add.fittedvalues
        df['ets_add_resid'] = df['Close'] - df['ets_additive']
    
    # Derived features
    df['ets_trend_agreement'] = (
        (df['ets_multiplicative'] > df['ets_multiplicative'].shift(1)) & 
        (df['ets_additive'] > df['ets_additive'].shift(1))
    ).astype(int)
    
    return df


df = add_ets_features(df)
print("✅ ETS features added")

✅ ETS features added


In [41]:
df.head()

,ds,Close,High,Low,Open,Volume,prophet_yhat,prophet_trend,prophet_upper,prophet_lower,prophet_diff,prophet_trend_strength,prophet_uncertainty,ets_multiplicative,ets_mult_resid,ets_additive,ets_add_resid,ets_trend_agreement
0,2023-05-18 00:00:00,1982.500000,1985.099976,1982.199951,1985.099976,0,1974.738266,1974.418647,2094.923607,1857.862280,7.761734,NaN,237.061327,1967.252856,15.247144,1966.381892,16.118108,0
1,2023-05-18 01:00:00,1980.800049,1982.500000,1980.000000,1982.500000,4884,1974.649736,1974.385135,2087.669784,1858.056739,6.150313,-0.033512,229.613044,1982.624862,-1.824813,1982.074581,-1.274532,1
2,2023-05-18 02:00:00,1979.900024,1984.199951,1979.599976,1980.800049,6725,1974.600533,1974.351623,2087.345511,1850.492201,5.299492,-0.033512,236.853310,1981.450286,-1.550262,1982.050658,-2.150633,0
3,2023-05-18 03:00:00,1977.000000,1982.000000,1976.000000,1979.800049,12013,1974.627209,1974.318110,2085.397727,1853.942031,2.372791,-0.033512,231.455696,1980.327383,-3.327383,1980.463266,-3.463266,0
4,2023-05-18 04:00:00,1977.900024,1978.699951,1974.599976,1977.099976,11054,1974.740508,1974.284598,2097.594833,1853.061922,3.159517,-0.033512,244.532911,1977.040111,0.859914,1977.822308,0.077716,0


In [42]:
print("\n🔧 Creating standard features...")

# Rename ds back to Datetime
df.rename(columns={"ds": "Datetime"}, inplace=True)
df['Datetime'] = pd.to_datetime(df['Datetime'])

# Returns
df['return'] = df['Close'].pct_change()
df['return_abs'] = df['return'].abs()
df['hl_spread'] = (df['High'] - df['Low']) / df['Close']

# EMAs
for window in [9, 12, 21, 26, 50]:
    df[f'ema_{window}'] = EMAIndicator(df['Close'], window=window).ema_indicator()

df['ema_fast'] = df['ema_12']
df['ema_slow'] = df['ema_26']
df['ema_cross'] = (df['ema_fast'] > df['ema_slow']).astype(int)
df['ema_distance'] = (df['ema_fast'] - df['ema_slow']) / df['Close']

# Price vs EMAs
df['price_vs_ema9'] = (df['Close'] - df['ema_9']) / df['Close']
df['price_vs_ema21'] = (df['Close'] - df['ema_21']) / df['Close']
df['price_vs_ema50'] = (df['Close'] - df['ema_50']) / df['Close']

# SMAs
for window in [20, 50, 100, 200]:
    df[f'sma_{window}'] = SMAIndicator(df['Close'], window=window).sma_indicator()

# Trend features - CRITICAL for detecting bullish/bearish
df['price_above_sma50'] = (df['Close'] > df['sma_50']).astype(int)
df['price_above_sma200'] = (df['Close'] > df['sma_200']).astype(int)
df['sma50_above_sma200'] = (df['sma_50'] > df['sma_200']).astype(int)  # Golden cross

# RSI
df['rsi'] = RSIIndicator(df['Close'], window=14).rsi()
df['rsi_ma'] = df['rsi'].rolling(5).mean()
df['rsi_overbought'] = (df['rsi'] > 70).astype(int)
df['rsi_oversold'] = (df['rsi'] < 30).astype(int)
df['rsi_bullish'] = (df['rsi'] > 50).astype(int)  # Above midline = bullish

# MACD
macd = MACD(df['Close'])
df['macd'] = macd.macd()
df['macd_signal'] = macd.macd_signal()
df['macd_histogram'] = df['macd'] - df['macd_signal']
df['macd_cross'] = (df['macd'] > df['macd_signal']).astype(int)
df['macd_bullish'] = (df['macd'] > 0).astype(int)  # Above zero = bullish

# ADX
adx = ADXIndicator(df['High'], df['Low'], df['Close'], window=14)
df['adx'] = adx.adx()
df['adx_pos'] = adx.adx_pos()
df['adx_neg'] = adx.adx_neg()
df['adx_bullish'] = (df['adx_pos'] > df['adx_neg']).astype(int)

# ATR
df['ATR'] = AverageTrueRange(df['High'], df['Low'], df['Close'], window=14).average_true_range()
df['atr_pct'] = df['ATR'] / df['Close']
df['atr_ma'] = df['ATR'].rolling(10).mean()

# Stochastic
stoch = StochasticOscillator(df['High'], df['Low'], df['Close'])
df['stoch_k'] = stoch.stoch()
df['stoch_d'] = stoch.stoch_signal()
df['stoch_bullish'] = (df['stoch_k'] > 50).astype(int)

# ROC
df['roc'] = ROCIndicator(df['Close'], window=12).roc()
df['roc_positive'] = (df['roc'] > 0).astype(int)

# Bollinger Bands
bb = BollingerBands(df['Close'], window=20, window_dev=2)
df['bb_high'] = bb.bollinger_hband()
df['bb_low'] = bb.bollinger_lband()
df['bb_mid'] = bb.bollinger_mavg()
df['bb_width'] = (df['bb_high'] - df['bb_low']) / df['bb_mid']
df['bb_position'] = (df['Close'] - df['bb_low']) / (df['bb_high'] - df['bb_low'])

# Momentum
df['price_momentum_12'] = df['Close'].pct_change(periods=12)
df['price_momentum_24'] = df['Close'].pct_change(periods=24)
df['momentum_positive'] = (df['price_momentum_12'] > 0).astype(int)

# Support/Resistance
df['resistance_20'] = df['High'].rolling(20).max()
df['support_20'] = df['Low'].rolling(20).min()

# Price position in range
df['price_in_range'] = (df['Close'] - df['support_20']) / (df['resistance_20'] - df['support_20'])

# Lag features
for lag in [1, 2, 3, 6, 12, 24]:
    df[f'return_lag_{lag}'] = df['return'].shift(lag)
    df[f'rsi_lag_{lag}'] = df['rsi'].shift(lag)

# Rolling statistics
df['return_std_12'] = df['return'].rolling(12).std()
df['return_mean_12'] = df['return'].rolling(12).mean()

# TREND SCORE - Critical feature!
df['trend_score'] = (
    df['price_above_sma50'] + 
    df['price_above_sma200'] + 
    df['sma50_above_sma200'] +
    df['ema_cross'] +
    df['macd_bullish'] +
    df['adx_bullish'] +
    df['rsi_bullish'] +
    df['momentum_positive'] +
    df['roc_positive']
)

# Time features
df['hour'] = df['Datetime'].dt.hour
df['day_of_week'] = df['Datetime'].dt.dayofweek
df['is_london_session'] = ((df['hour'] >= 8) & (df['hour'] <= 16)).astype(int)
df['is_ny_session'] = ((df['hour'] >= 13) & (df['hour'] <= 21)).astype(int)
df['is_asian_session'] = ((df['hour'] >= 0) & (df['hour'] <= 8)).astype(int)

df = df.dropna().reset_index(drop=True)
print(f"After features: {len(df)} rows with {len(df.columns)} columns")


🔧 Creating standard features...
After features: 13560 rows with 93 columns


In [43]:
def create_signal_labels_precise(df, horizon=6, threshold=0.01, rr_ratio=2.0):
    df = df.copy().reset_index(drop=True)
    risk_mult = 1.5
    
    df['potential_entry'] = df['Close']
    df['potential_sl_buy'] = df['Close'] - risk_mult * df['ATR']
    df['potential_tp_buy'] = df['Close'] + rr_ratio * risk_mult * df['ATR']
    df['potential_sl_sell'] = df['Close'] + risk_mult * df['ATR']
    df['potential_tp_sell'] = df['Close'] - rr_ratio * risk_mult * df['ATR']
    
    df['future_return'] = (df['Close'].shift(-horizon) - df['Close']) / df['Close']
    
    signal = np.zeros(len(df), dtype=int)
    
    for i in range(len(df) - horizon):
        entry = df.loc[i, 'potential_entry']
        future_return = df.loc[i, 'future_return']
        
        future = df.loc[i+1:i+horizon]
        if future.empty:
            continue
        
        # Check BUY
        sl_buy = df.loc[i, 'potential_sl_buy']
        tp_buy = df.loc[i, 'potential_tp_buy']
        
        tp_hit = (future['High'] >= tp_buy)
        sl_hit = (future['Low'] <= sl_buy)
        
        if tp_hit.any():
            tp_idx = future[tp_hit].index[0]
            if sl_hit.any():
                sl_idx = future[sl_hit].index[0]
                if tp_idx < sl_idx and future_return > threshold:
                    signal[i] = 1
            else:
                if future_return > threshold:
                    signal[i] = 1
        
        # Check SELL
        if signal[i] == 0:
            sl_sell = df.loc[i, 'potential_sl_sell']
            tp_sell = df.loc[i, 'potential_tp_sell']
            
            tp_hit = (future['Low'] <= tp_sell)
            sl_hit = (future['High'] >= sl_sell)
            
            if tp_hit.any():
                tp_idx = future[tp_hit].index[0]
                if sl_hit.any():
                    sl_idx = future[sl_hit].index[0]
                    if tp_idx < sl_idx and future_return < -threshold:
                        signal[i] = 2
                else:
                    if future_return < -threshold:
                        signal[i] = 2
    
    df['signal'] = signal
    return df

df = create_signal_labels_precise(df, horizon=6, threshold=0.01, rr_ratio=2.0)

print("Signal distribution:")
print(df['signal'].value_counts())
print("\nSignal distribution (%):")
print(df['signal'].value_counts(normalize=True) * 100)

Signal distribution:
signal
0    13026
1      288
2      246
Name: count, dtype: int64

Signal distribution (%):
signal
0    96.061947
1     2.123894
2     1.814159
Name: proportion, dtype: float64


In [44]:
exclude_cols = ['Datetime', 'signal', 'potential_entry', 'potential_sl_buy', 
                'potential_tp_buy', 'potential_sl_sell', 'potential_tp_sell',
                'future_return']

feature_cols = [col for col in df.columns if col not in exclude_cols]
print(f"\n📦 Using {len(feature_cols)} features")

train_size = int(0.75 * len(df))
train = df.iloc[:train_size].copy()
test = df.iloc[train_size:].copy()

X_train = train[feature_cols]
y_train = train['signal']
X_test = test[feature_cols]
y_test = test['signal']

print(f"Training: {len(X_train)} samples")
print(f"Test: {len(X_test)} samples")
print("\nTraining signals:")
print(y_train.value_counts())


📦 Using 92 features
Training: 10170 samples
Test: 3390 samples

Training signals:
signal
0    9845
1     176
2     149
Name: count, dtype: int64


In [45]:
print("\n🤖 Training models...")

class_counts = y_train.value_counts().sort_index()
total = len(y_train)
class_weights = {cls: total / (len(class_counts) * count) for cls, count in class_counts.items()}

# Adjust weights
class_weights[1] = class_weights[1] * 2.0
class_weights[2] = class_weights[2] * 2.0

sample_weights = y_train.map(class_weights)

# XGBoost
xgb_model = XGBClassifier(
    n_estimators=1200,
    max_depth=3,
    learning_rate=0.01,
    subsample=0.7,
    colsample_bytree=0.6,
    min_child_weight=10,
    gamma=0.5,
    reg_alpha=1.0,
    reg_lambda=2.0,
    random_state=42,
    objective='multi:softmax',
    num_class=3,
    scale_pos_weight=2.0
)

xgb_model.fit(X_train, y_train, sample_weight=sample_weights)

# LightGBM
lgbm_model = LGBMClassifier(
    n_estimators=1200,
    max_depth=3,
    learning_rate=0.01,
    subsample=0.7,
    colsample_bytree=0.6,
    min_child_samples=30,
    reg_alpha=1.0,
    reg_lambda=2.0,
    random_state=42,
    verbosity=-1
)

lgbm_model.fit(X_train, y_train, sample_weight=sample_weights)

# Random Forest
rf_model = RandomForestClassifier(
    n_estimators=600,
    max_depth=6,
    min_samples_split=30,
    min_samples_leaf=15,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train, sample_weight=sample_weights)


🤖 Training models...


,n_estimators,600
,criterion,'gini'
,max_depth,6
,min_samples_split,30
,min_samples_leaf,15
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [46]:
print("\n📈 Evaluating...")

xgb_proba = xgb_model.predict_proba(X_test)
lgbm_proba = lgbm_model.predict_proba(X_test)
rf_proba = rf_model.predict_proba(X_test)

ensemble_proba = (0.6 * xgb_proba + 0.25 * lgbm_proba + 0.15 * rf_proba)

# Find thresholds
def find_threshold(y_true, y_proba, signal_class, target_precision=0.80):
    results = []
    for thresh in np.linspace(0.50, 0.95, 200):
        y_pred = np.zeros(len(y_true))
        y_pred[y_proba[:, signal_class] >= thresh] = signal_class
        
        mask = (y_true == signal_class) | (y_pred == signal_class)
        if mask.sum() == 0:
            continue
        
        if (y_pred == signal_class).sum() == 0:
            continue
        
        precision = precision_score(y_true[mask], y_pred[mask], 
                                   labels=[signal_class], average='micro', zero_division=0)
        recall = recall_score(y_true == signal_class, y_pred == signal_class, zero_division=0)
        
        results.append({
            'threshold': thresh,
            'precision': precision,
            'recall': recall,
            'signals': (y_pred == signal_class).sum()
        })
    
    results_df = pd.DataFrame(results)
    good = results_df[results_df['precision'] >= target_precision]
    
    if len(good) == 0:
        best = results_df.loc[results_df['precision'].idxmax()]
        return best['threshold'], best.to_dict(), False
    
    best = good.loc[good['recall'].idxmax()]
    return best['threshold'], best.to_dict(), True

buy_threshold, buy_metrics, buy_success = find_threshold(y_test, ensemble_proba, 1)
sell_threshold, sell_metrics, sell_success = find_threshold(y_test, ensemble_proba, 2)

print(f"\n🟢 BUY: Precision={buy_metrics['precision']*100:.1f}%, Recall={buy_metrics['recall']*100:.1f}%, Signals={int(buy_metrics['signals'])}")
print(f"🔴 SELL: Precision={sell_metrics['precision']*100:.1f}%, Recall={sell_metrics['recall']*100:.1f}%, Signals={int(sell_metrics['signals'])}")


📈 Evaluating...

🟢 BUY: Precision=66.7%, Recall=3.6%, Signals=6
🔴 SELL: Precision=5.3%, Recall=6.2%, Signals=113


In [ ]:


# =============================================
# 8. EVALUATE & SAVE
# =============================================
print("\n📈 Evaluating...")

xgb_proba = xgb_model.predict_proba(X_test)
lgbm_proba = lgbm_model.predict_proba(X_test)
rf_proba = rf_model.predict_proba(X_test)

ensemble_proba = (0.6 * xgb_proba + 0.25 * lgbm_proba + 0.15 * rf_proba)

# Find thresholds
def find_threshold(y_true, y_proba, signal_class, target_precision=0.80):
    results = []
    for thresh in np.linspace(0.50, 0.95, 200):
        y_pred = np.zeros(len(y_true))
        y_pred[y_proba[:, signal_class] >= thresh] = signal_class
        
        mask = (y_true == signal_class) | (y_pred == signal_class)
        if mask.sum() == 0:
            continue
        
        if (y_pred == signal_class).sum() == 0:
            continue
        
        precision = precision_score(y_true[mask], y_pred[mask], 
                                   labels=[signal_class], average='micro', zero_division=0)
        recall = recall_score(y_true == signal_class, y_pred == signal_class, zero_division=0)
        
        results.append({
            'threshold': thresh,
            'precision': precision,
            'recall': recall,
            'signals': (y_pred == signal_class).sum()
        })
    
    results_df = pd.DataFrame(results)
    good = results_df[results_df['precision'] >= target_precision]
    
    if len(good) == 0:
        best = results_df.loc[results_df['precision'].idxmax()]
        return best['threshold'], best.to_dict(), False
    
    best = good.loc[good['recall'].idxmax()]
    return best['threshold'], best.to_dict(), True

buy_threshold, buy_metrics, buy_success = find_threshold(y_test, ensemble_proba, 1)
sell_threshold, sell_metrics, sell_success = find_threshold(y_test, ensemble_proba, 2)

print(f"\n🟢 BUY: Precision={buy_metrics['precision']*100:.1f}%, Recall={buy_metrics['recall']*100:.1f}%, Signals={int(buy_metrics['signals'])}")
print(f"🔴 SELL: Precision={sell_metrics['precision']*100:.1f}%, Recall={sell_metrics['recall']*100:.1f}%, Signals={int(sell_metrics['signals'])}")

# Save model
import joblib

model_data = {
    'xgb_model': xgb_model,
    'lgbm_model': lgbm_model,
    'rf_model': rf_model,
    'feature_cols': feature_cols,
    'buy_threshold': buy_threshold,
    'sell_threshold': sell_threshold,
    'ensemble_weights': [0.6, 0.25, 0.15],
    'params': {
        'horizon': 6,
        'threshold': 0.01,
        'rr_ratio': 2.0,
        'risk_mult': 1.5
    }
}

joblib.dump(model_data, '/Users/mac/Desktop/Upwork/GoldSignalGenerator/Artifacts/GLD_v2_prophet.pkl')
print("\n💾 Model saved as GLD_v2_prophet.pkl")
print("✅ Training complete with Prophet & ETS features!")